# PyCAM-SIMA Dask checkpoint fan-out

This standalone Notebook demonstrates task-oriented execution without a persistent socket worker. Inside a PBS allocation, Dask launches each 24-rank MPI segment directly in that one allocation. Outside an allocation it retains the original per-segment PBS submission mode. A common immutable Python-owned checkpoint Future feeds independent experiment branches, and a serializable `SegmentPlan` can run phase, scheme, observation, and complete-step actions inside one segment.

## 1. Execution model

```text
Jupyter/PBS controller + local Dask Client
           │
           ├── base MPI segment: 24 ranks × 10 steps
           │                    │
           │                    └── immutable checkpoint Future
           │                                  │
           ├──────────────────────────────────┼── control:    5 steps
           ├──────────────────────────────────└── no-kessler: 5 steps
           └── warm-initial: +1 K, 0 steps
                         │ exact edit check
                         └── warm: 5 steps
```

The base process exits after writing its checkpoint. Each segment restores private NumPy arrays and creates a new `MPI.COMM_WORLD`; this is checkpoint/restart fan-out, not operating-system `fork()`. With `execution_mode='allocation'`, all segments report the same outer `PBS_JOBID` and no branch calls `qsub`. Full-node MPI launches are serialized. The zero-step `warm-initial` checkpoint separates verification of the requested edit from the later model response.

## 2. Configure one experiment

Run this cell again to create a fresh timestamp before repeating the experiment. Existing branch directories are deliberately never overwritten.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from netCDF4 import Dataset
from pycam_sima import (
    BranchSpec,
    DaskExperimentClient,
    FieldEdit,
    ObserveFields,
    RunPhase,
    RunScheme,
    SegmentPlan,
)

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/dask_notebook_trials' / f'fanout-{stamp}'
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'branches'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
dask_workers = 1 if execution_mode == 'allocation' else 3

print('pycam_sima', pycam_sima.__version__)
print('experiment root:', experiment_root)
print('execution mode:', execution_mode)

pycam_sima 0.8.0
experiment root: /glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116
execution mode: allocation


## 3. Create the Dask controller

The Dask workers orchestrate 24-rank MPI segments. In `allocation` mode one worker serially launches every segment with `mpiexec` inside the current PBS job. In `pbs` mode three workers may independently submit branch PBS jobs.

In [2]:
if 'client' in globals():
    client.close()

client = Client(
    processes=False,
    n_workers=dask_workers,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.14.4.234:8787/status,
Dashboard: http://10.14.4.234:8787/status,Workers: 1
Total threads: 1,Total memory: 80.00 GiB
Status: running,Using processes: False
Comm: inproc://10.14.4.234/160601/1,Workers: 0
Dashboard: http://10.14.4.234:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.14.4.234/160601/4,Total threads: 1
Dashboard: http://10.14.4.234:36387/status,Memory: 80.00 GiB
Nanny: None,


## 4. Submit the common base and the zero-step warm edit

This first stage runs two Dask-managed MPI segments. The base runs 10 steps. `warm-initial` restores that exact state, adds `1 K` to `air_temperature`, runs zero model steps, and writes another checkpoint. In allocation mode neither segment submits another PBS job.

In [3]:
base = experiments.submit_base(
    BranchSpec('base', steps=10)
)

warm_initial = experiments.submit_branch(
    base,
    BranchSpec(
        'warm-initial',
        steps=0,
        field_edits=(
            FieldEdit('air_temperature', 'add', 1.0),
        ),
    ),
)

initial_summaries = experiments.summaries({
    'base': base,
    'warm-initial': warm_initial,
})
initial_summaries

{'base': {'branch': 'base',
  'parent_branch': None,
  'step': 10,
  'history_samples': 11,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/base/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/base/history',
  'checkpoint_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/base/checkpoint',
  'snapshot_nbytes': 37599210,
  'execution_mode': 'allocation',
  'pbs_job_id': '6857641.desched1',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_base_78da2c8b6f99.log'},
 'warm-initial': {'branch': 'warm-initial',
  'parent_branch': 'base',
  'step': 10,
  'history_samples': 11,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/warm-initial/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-2

## 5. Verify the edit before model evolution

Both checkpoints are at model step 10. This comparison therefore tests only `FieldEdit('air_temperature', 'add', 1.0)` across all 24 ranks. Every edited array must match NumPy's elementwise `base + 1.0` operation bit for bit. Recomputing `(base + 1.0) - base` performs another floating-point operation, so that diagnostic is required to equal `1 K` only within one spacing of the largest base value.

In [4]:
def checkpoint_field(summary_map, branch, field, rank=0):
    checkpoint_file = (
        Path(summary_map[branch]['checkpoint_dir'])
        / f'rank-{rank:03d}.npz'
    )
    with np.load(checkpoint_file, allow_pickle=False) as arrays:
        return arrays[field].copy()

base_checkpoint = Path(initial_summaries['base']['checkpoint_dir'])
rank_count = len(tuple(base_checkpoint.glob('rank-*.npz')))
base_temperatures = [
    checkpoint_field(initial_summaries, 'base', 'air_temperature', rank)
    for rank in range(rank_count)
]
warm_initial_temperatures = [
    checkpoint_field(
        initial_summaries, 'warm-initial', 'air_temperature', rank
    )
    for rank in range(rank_count)
]
initial_differences = [
    warm - base
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
]
exact_numpy_edit = all(
    np.array_equal(warm, np.add(base, 1.0))
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
)
maximum_roundoff_from_1K = float(
    max(np.abs(delta - 1.0).max() for delta in initial_differences)
)
roundoff_tolerance = float(
    max(np.spacing(np.abs(base).max()) for base in base_temperatures)
)
difference_within_roundoff = (
    maximum_roundoff_from_1K <= roundoff_tolerance
)

assert exact_numpy_edit
assert difference_within_roundoff
sample_count = sum(delta.size for delta in initial_differences)
{
    'ranks': rank_count,
    'rank_local_shape': initial_differences[0].shape,
    'difference_min': float(min(delta.min() for delta in initial_differences)),
    'difference_max': float(max(delta.max() for delta in initial_differences)),
    'difference_mean': float(
        sum(delta.sum() for delta in initial_differences) / sample_count
    ),
    'exact_numpy_add_1K': exact_numpy_edit,
    'maximum_roundoff_from_1K': maximum_roundoff_from_1K,
    'roundoff_tolerance': roundoff_tolerance,
    'difference_within_roundoff': difference_within_roundoff,
}

{'ranks': 24,
 'rank_local_shape': (4, 4, 30, 3, 3),
 'difference_min': 0.9999999999999716,
 'difference_max': 1.0000000000000284,
 'difference_mean': 1.0,
 'exact_numpy_add_1K': True,
 'maximum_roundoff_from_1K': 2.842170943040401e-14,
 'roundoff_tolerance': 5.684341886080802e-14,
 'difference_within_roundoff': True}

## 6. Continue the three five-step experiments

`control` and `no-kessler` continue directly from the base checkpoint. `warm` continues from the already verified `warm-initial` checkpoint without applying a second edit. This stage creates three additional Dask tasks; allocation mode executes their full-node MPI segments serially in the same PBS job.

In [5]:
branches = experiments.fork(
    base,
    (
        BranchSpec('control', steps=5),
        BranchSpec(
            'no-kessler',
            steps=5,
            disable_schemes=('kessler',),
        ),
    ),
)
branches['warm'] = experiments.submit_branch(
    warm_initial,
    BranchSpec('warm', steps=5),
)

summaries = experiments.summaries(branches)
summaries

{'control': {'branch': 'control',
  'parent_branch': 'base',
  'step': 15,
  'history_samples': 16,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/control/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/control/history',
  'checkpoint_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/control/checkpoint',
  'snapshot_nbytes': 37599210,
  'execution_mode': 'allocation',
  'pbs_job_id': '6857641.desched1',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_control_58d917b3c9b9.log'},
 'no-kessler': {'branch': 'no-kessler',
  'parent_branch': 'base',
  'step': 15,
  'history_samples': 16,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/no-kessler/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_t

## 7. What the summary tells you

All three final branches report `step=15` and 16 history samples (the initial state plus 15 completed steps). `control` and `no-kessler` have `parent_branch='base'`; `warm` has `parent_branch='warm-initial'`. The summary also contains `execution_mode`, each PBS job ID, run/history/checkpoint/log paths, and serialized checkpoint size without downloading the full checkpoint Future. In allocation mode every PBS job ID must be identical.

In [6]:
[
    {
        'branch': name,
        'step': summary['step'],
        'history_samples': summary['history_samples'],
        'execution_mode': summary['execution_mode'],
        'pbs_job_id': summary['pbs_job_id'],
        'checkpoint_GiB': summary['snapshot_nbytes'] / 1024**3,
        'history_dir': summary['history_dir'],
        'log_path': summary['log_path'],
    }
    for name, summary in summaries.items()
]

[{'branch': 'control',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6857641.desched1',
  'checkpoint_GiB': 0.035016993060708046,
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/control/history',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_control_58d917b3c9b9.log'},
 {'branch': 'no-kessler',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6857641.desched1',
  'checkpoint_GiB': 0.03501703776419163,
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116/branches/no-kessler/history',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_no-kessler_49c7f73f158b.log'},
 {'branch': 'warm',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6857641.desched1',
  'checkpoint_GiB': 0.035016993060708046,
  'histor

## 8. Compare the temperature after five model steps

Every branch checkpoint contains the complete 214-field StatePool for all 24 ranks. Unlike the exact pre-run edit, the warm-control difference after five nonlinear model steps is expected to vary around `1 K`. The deviation from `1 K`, rather than the total warm-control difference, measures the subsequent model response.

In [7]:
control_temperature = checkpoint_field(
    summaries, 'control', 'air_temperature'
)
warm_temperature = checkpoint_field(
    summaries, 'warm', 'air_temperature'
)
temperature_difference = warm_temperature - control_temperature

{
    'rank': 0,
    'shape': control_temperature.shape,
    'difference_min': float(temperature_difference.min()),
    'difference_max': float(temperature_difference.max()),
    'difference_mean': float(temperature_difference.mean()),
    'maximum_deviation_from_1K': float(
        np.abs(temperature_difference - 1.0).max()
    ),
    'bitwise_identical': bool(np.array_equal(control_temperature, warm_temperature)),
}

{'rank': 0,
 'shape': (4, 4, 30, 3, 3),
 'difference_min': 0.9913538548482279,
 'difference_max': 1.0004294651569978,
 'difference_mean': 0.9997454265889733,
 'maximum_deviation_from_1K': 0.008646145151772089,
 'bitwise_identical': False}

## 9. Inspect a global field after every model step

Each branch history directory inherits the 10 base timestamps and adds 5 branch timestamps. These NetCDF files contain the 26 configured global diagnostics. Unlike the final rank-local checkpoint, this gives one global field value at every completed model step.

In [8]:
def history_statistics(branch, variable):
    summary = summaries[branch]
    history_dir = summary.get(
        'history_dir',
        Path(summary['checkpoint_dir']).parent / 'history',
    )
    files = sorted(Path(history_dir).glob('*.nc'))
    records = []
    for path in files:
        with Dataset(path) as dataset:
            values = np.asarray(dataset[variable][0])
            records.append({
                'step': int(dataset['nsteph'][0]),
                'file': path.name,
                'minimum': float(values.min()),
                'maximum': float(values.max()),
                'mean': float(values.mean()),
            })
    return records

control_rain_by_step = history_statistics('control', 'RAINQM')
no_kessler_rain_by_step = history_statistics('no-kessler', 'RAINQM')

{
    'control_last': control_rain_by_step[-1],
    'no_kessler_last': no_kessler_rain_by_step[-1],
    'control_all_steps': control_rain_by_step,
}

{'control_last': {'step': 15,
  'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-27000i.nc',
  'minimum': 0.0,
  'maximum': 0.0,
  'mean': 0.0},
 'no_kessler_last': {'step': 15,
  'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-27000i.nc',
  'minimum': 0.0,
  'maximum': 0.0,
  'mean': 0.0},
 'control_all_steps': [{'step': 0,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-00000i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 1,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-01800i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 2,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-03600i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 3,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-05400i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 4,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-0720

## 10. Run phase and scheme actions

A `SegmentPlan` executes all actions inside one MPI segment, so its actions share one live StatePool without writing an intermediate checkpoint. `submit_action()` instead creates one new Dask task and one final checkpoint, which is useful when that action boundary must become a Future or fork point. Standalone phase and scheme calls are intentionally marked `unsafe=True`: they do not advance the model clock or automatically run prerequisite calculations.

In [ ]:
granular_plan = SegmentPlan(
    'granular-kessler-then-map',
    actions=(
        RunScheme(
            'kessler',
            group='physics_before_coupler',
        ),
        ObserveFields((
            'potential_temperature',
            'large_scale_precipitation_rate',
        )),
        RunPhase('dynamics_to_physics'),
        ObserveFields(('air_temperature',)),
    ),
    unsafe=True,
)

granular = experiments.submit_plan(base, granular_plan)
granular_summary = experiments.summary(granular).result()
rank0_potential_temperature = experiments.field(
    granular,
    'potential_temperature',
    rank=0,
).result()

{
    'step': granular_summary['step'],
    'action_trace': granular_summary['action_trace'],
    'rank0_field_shape': rank0_potential_temperature.shape,
    'rank0_field_mean': float(rank0_potential_temperature.mean()),
}

### Make one scheme boundary a separate Future

This form starts a new 24-rank MPI segment, restores `base`, runs exactly one scheme, and writes a checkpoint. A later `submit_action()`, `submit_plan()`, or `fork()` can use `kessler_only` as its parent.

In [ ]:
kessler_only = experiments.submit_action(
    base,
    name='single-kessler-action',
    action=RunScheme(
        'kessler',
        group='physics_before_coupler',
    ),
)
experiments.summary(kessler_only).result()

In [9]:
client.close()
print('Dask client closed; PBS results remain under', experiment_root)

Dask client closed; PBS results remain under /glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260722-225116


In [1]:
import os
import socket
import sys

print("host:", socket.gethostname())
print("python:", sys.executable)
print("PBS_JOBID:", os.environ.get("PBS_JOBID"))
print("PBS_NODEFILE:", os.environ.get("PBS_NODEFILE"))

host: dec2448
python: /glade/work/ruitong/pycam-sima/.venv/bin/python
PBS_JOBID: 6857641.desched1
PBS_NODEFILE: /var/spool/pbs/aux/6857641.desched1
